# Tutorial 54: S3 the native way

The scenario of tutorial 15 (per-sector ticker summary), but written the way
notebooks read S3 every day — a literal `s3://` URL right in the cell, no
registered data source, no mount paths:

1. the `%%krauncher` magic detects the literal object URL, pre-fetches it via
   the data bridge **before** the container starts (IO lands in the measured
   download phase) and rewrites the literal to the local copy;
2. credentials come from your account defaults (Storage → Credentials → S3) —
   nothing travels through the cell code;
3. the client cannot size a private object — `--dataset-size` declares it for
   the quote;
4. results come back as notebook values (no output source in this variant).

**Replace `s3://YOUR-BUCKET/tickers.csv` with a real object before running**
(tutorial 15's registered `source-data` points at the same kind of CSV).


In [ ]:
%load_ext krauncher_magic

In [ ]:
%env CAS_CLIENT_CONFIG=../../cas-client/.env

In [ ]:
%%krauncher --dataset-size 1
import csv

# Literal s3:// object — auto-detected, pre-fetched, read locally.
_rows = []
with open("s3://YOUR-BUCKET/tickers.csv") as f:
    for row in csv.DictReader(f):
        _rows.append(dict(row))

sectors = {}
for row in _rows:
    s = row["sector"]
    sectors[s] = sectors.get(s, 0) + 1

top_sectors = dict(sorted(sectors.items(), key=lambda x: x[1], reverse=True)[:10])
total_tickers = len(_rows)
sector_count = len(sectors)
print(f"{total_tickers} tickers, {sector_count} sectors")


In [ ]:
print(f"Total tickers: {total_tickers}")
print(f"Sectors found: {sector_count}")
for sector, count in top_sectors.items():
    print(f"  {sector}: {count}")
